In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
from scipy import stats
from sklearn.preprocessing import StandardScaler
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family']        = 'serif'
plt.rcParams['font.serif']         = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['axes.linewidth']     = 1.5
plt.rcParams['xtick.major.width']  = 1.5
plt.rcParams['ytick.major.width']  = 1.5

DATA_PATH     = 'B0005_特征数据集.xlsx'
TRAIN_RATIO   = 0.6
FEATURES      = ['VDET', 'CCDT', 'HDTT', 'ICP']
TARGET_COL    = 'SOH'
OUTPUT_PREFIX = 'B0005'

ENSEMBLE_SIZE = 100
NOISE_STD     = 0.2
MAX_IMF       = 6
MAX_SIFT      = 100
SD_THRESHOLD  = 0.2

LB_LAGS  = 10      
LB_ALPHA = 0.05    
WH_CONF  = 0.95    

# ══════════════════════════════════════════════════════════
# 1. ICEEMDAN
# ══════════════════════════════════════════════════════════
def find_extrema(signal):
    n = len(signal)
    max_idx = [i for i in range(1, n-1)
               if signal[i] > signal[i-1] and signal[i] > signal[i+1]]
    min_idx = [i for i in range(1, n-1)
               if signal[i] < signal[i-1] and signal[i] < signal[i+1]]
    return np.array(max_idx), np.array(min_idx)

def envelope_mean(signal):
    n = len(signal)
    t = np.arange(n)
    max_idx, min_idx = find_extrema(signal)
    if len(max_idx) < 2 or len(min_idx) < 2:
        return np.zeros(n)
    max_idx_ext = np.r_[2*max_idx[0]-max_idx[1], max_idx, 2*max_idx[-1]-max_idx[-2]]
    min_idx_ext = np.r_[2*min_idx[0]-min_idx[1], min_idx, 2*min_idx[-1]-min_idx[-2]]
    max_val_ext = np.r_[signal[max_idx[0]], signal[max_idx], signal[max_idx[-1]]]
    min_val_ext = np.r_[signal[min_idx[0]], signal[min_idx], signal[min_idx[-1]]]
    try:
        cs_max = CubicSpline(max_idx_ext, max_val_ext)
        cs_min = CubicSpline(min_idx_ext, min_val_ext)
        upper  = cs_max(t)
        lower  = cs_min(t)
    except Exception:
        return np.zeros(n)
    return (upper + lower) / 2.0

def emd_first_imf(signal, max_sift=MAX_SIFT, sd_thresh=SD_THRESHOLD):
    h = signal.copy()
    for _ in range(max_sift):
        m     = envelope_mean(h)
        h_new = h - m
        denom = np.sum(h ** 2)
        if denom < 1e-10:
            break
        sd = np.sum((h - h_new) ** 2) / denom
        h  = h_new
        if sd < sd_thresh:
            break
    return h

def iceemdan(signal, I=ENSEMBLE_SIZE, eps=NOISE_STD,
             max_imf=MAX_IMF, max_sift=MAX_SIFT,
             sd_thresh=SD_THRESHOLD, seed=42):
    rng     = np.random.RandomState(seed)
    x       = signal.copy()
    n       = len(x)
    std_x   = np.std(x)
    if std_x < 1e-10:
        return [x]
    imfs    = []
    residue = x.copy()
    for k in range(max_imf):
        imf_ensemble = np.zeros(n)
        for i in range(I):
            noise     = rng.randn(n) * eps * std_x
            noisy_sig = residue + noise
            imf_i     = emd_first_imf(noisy_sig, max_sift=max_sift, sd_thresh=sd_thresh)
            imf_ensemble += imf_i
        imf_k   = imf_ensemble / I
        residue = residue - imf_k
        imfs.append(imf_k)
        max_idx, min_idx = find_extrema(residue)
        if len(max_idx) + len(min_idx) < 4:
            break
    imfs.append(residue)
    return imfs

# ══════════════════════════════════════════════════════════
# 2. 统计验证函数
# ══════════════════════════════════════════════════════════

def ljungbox_test(x, lags=LB_LAGS):
    n    = len(x)
    xd   = x - np.mean(x)
    c0   = np.dot(xd, xd) / n        
    Q = 0.0
    for k in range(1, lags + 1):
        ck  = np.dot(xd[k:], xd[:-k]) / n
        r_k = ck / (c0 + 1e-12)
        Q  += r_k ** 2 / (n - k)
    Q  *= n * (n + 2)
    p   = 1 - stats.chi2.cdf(Q, df=lags)
    return float(Q), float(p)

def wh_energy_test(imfs, conf=WH_CONF):
    n_imf    = len(imfs) - 1  
    energies = []
    periods  = []
    for imf in imfs[:-1]:
        e_k = np.mean(imf ** 2)
        energies.append(max(e_k, 1e-12))
        zc  = np.where(np.diff(np.sign(imf)))[0]
        t_k = 2 * np.mean(np.diff(zc)) if len(zc) >= 2 else float(len(imf))
        periods.append(max(t_k, 1.0))
    ln_E = np.array([np.log(e) for e in energies])
    ln_T = np.array([np.log(t) for t in periods])
    if len(ln_T) >= 2:
        slope, intercept, _, _, _ = stats.linregress(ln_T, ln_E)
    else:
        slope, intercept = -1.0, 0.0
    z = stats.norm.ppf((1 + conf) / 2)
    rows = []
    for k in range(n_imf):
        baseline = slope * ln_T[k] + intercept
        n_eff    = max(len(imfs[k]) / periods[k], 2)
        sigma_k  = np.sqrt(2.0 / n_eff)
        upper_ci = baseline + z * sigma_k
        lower_ci = baseline - z * sigma_k
        in_band  = (lower_ci <= ln_E[k] <= upper_ci)
        is_signal = (ln_E[k] > upper_ci)
        rows.append({
            'IMF':       f'IMF{k+1}',
            'Energy':    round(energies[k], 6),
            'Period':    round(periods[k],  2),
            'ln_E':      round(ln_E[k],     4),
            'ln_T':      round(ln_T[k],     4),
            'Baseline':  round(baseline,    4),
            'Upper_CI':  round(upper_ci,    4),
            'Lower_CI':  round(lower_ci,    4),
            'In_Band':   in_band,
            'WH_Signal': is_signal,
        })
    rows.append({
        'IMF': 'Residual', 'Energy': round(np.mean(imfs[-1]**2), 6),
        'Period': len(imfs[-1]), 'ln_E': None, 'ln_T': None,
        'Baseline': None, 'Upper_CI': None, 'Lower_CI': None,
        'In_Band': False, 'WH_Signal': True,
    })
    return pd.DataFrame(rows)


def combined_validation(imfs):
    wh_df = wh_energy_test(imfs)
    lb_rows = []
    for k, imf in enumerate(imfs[:-1]):
        Q, p    = ljungbox_test(imf)
        is_sig  = (p <= LB_ALPHA)
        lb_rows.append({'IMF': f'IMF{k+1}', 'LB_Stat': round(Q, 4),
                        'p_value': round(p, 6), 'LB_Signal': is_sig})
    lb_rows.append({'IMF': 'Residual', 'LB_Stat': None,
                    'p_value': None, 'LB_Signal': True})
    lb_df = pd.DataFrame(lb_rows)
    merged = wh_df.merge(lb_df, on='IMF')
    merged['Keep']     = merged['WH_Signal'] | merged['LB_Signal']
    merged.loc[merged['IMF'] == 'Residual', 'Keep'] = True
    merged['Decision'] = merged['Keep'].map({True: 'Keep', False: 'Noise'})
    return merged

# ══════════════════════════════════════════════════════════
# 3. 加载数据 & 分解
# ══════════════════════════════════════════════════════════
df        = pd.read_excel(DATA_PATH)
cycle_col = df.columns[0]
split_idx      = int(len(df) * TRAIN_RATIO)
train_df       = df.iloc[:split_idx].copy()
test_df        = df.iloc[split_idx:].copy()
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(train_df[FEATURES])
X_test_scaled  = scaler.transform(test_df[FEATURES])
print(f"总循环数: {len(df)}  |  训练集: {len(train_df)}  |  测试集: {len(test_df)}\n")

results = {}
for feat_idx, feat in enumerate(FEATURES):
    print(f"[{feat_idx+1}/{len(FEATURES)}] 正在分解特征: {feat} ...")
    signal = X_train_scaled[:, feat_idx]
    imfs   = iceemdan(signal, I=ENSEMBLE_SIZE, eps=NOISE_STD,
                      max_imf=MAX_IMF, seed=42)
    results[feat] = {'imfs': imfs, 'n_imfs': len(imfs), 'signal': signal}
    print(f"  → 分解得到 {len(imfs)} 个分量（含残差）")

# ══════════════════════════════════════════════════════════
# 4. 统计验证
# ══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  统计验证：Wu-Huang 能量检验 + Ljung-Box 白噪声检验")
print("=" * 60)

val_results = {}
for feat in FEATURES:
    imfs      = results[feat]['imfs']
    val_df    = combined_validation(imfs)
    val_results[feat] = val_df

    kept    = val_df[val_df['Keep']]['IMF'].tolist()
    dropped = val_df[~val_df['Keep']]['IMF'].tolist()
    print(f"\n── {feat} ──")
    show_cols = ['IMF', 'WH_Signal', 'LB_Stat', 'p_value', 'LB_Signal',
                 'Keep', 'Decision']
    print(val_df[show_cols].to_string(index=False))
    print(f"  ✅ 保留: {kept}")
    if dropped:
        print(f"  ❌ 剔除: {dropped}")

# ══════════════════════════════════════════════════════════
# 5. 可视化
# ══════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(16, 10), facecolor='white')
panel_labels = ['(a)', '(b)', '(c)', '(d)']
for ax, feat, plabel in zip(axes.flat, FEATURES, panel_labels):
    val_df   = val_results[feat]
    imf_rows = val_df[val_df['IMF'] != 'Residual'].copy()
    x_pos    = np.arange(len(imf_rows))
    x_labels = imf_rows['IMF'].tolist()

    bar_colors = ['#E53935' if s else '#43A047'
                  for s in imf_rows['LB_Signal'].tolist()]
    ax.bar(x_pos, imf_rows['p_value'].astype(float).values,
           color=bar_colors, alpha=0.75, width=0.45, zorder=2,
           label='LB p-value')
    ax.axhline(LB_ALPHA, color='#E53935', linestyle='--',
               linewidth=1.5, label=f'α = {LB_ALPHA}')
    y_max = max(imf_rows['p_value'].astype(float).max() * 1.35, LB_ALPHA * 2)
    ax.set_ylim(0, y_max)
    ax.set_ylabel('Ljung–Box  p-value', fontsize=14)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, fontsize=14)
    ax.tick_params(axis='y', labelsize=12)

    ax2 = ax.twinx()
    ln_E_vals = imf_rows['ln_E'].astype(float).values
    base_vals = imf_rows['Baseline'].astype(float).values
    uci_vals  = imf_rows['Upper_CI'].astype(float).values
    lci_vals  = imf_rows['Lower_CI'].astype(float).values

    ax2.plot(x_pos, ln_E_vals, 'o-', color='#1565C0',
             linewidth=2, markersize=7, zorder=3, label='ln(Energy)')
    ax2.plot(x_pos, base_vals, '--', color='#757575',
             linewidth=1.5, label='WH Baseline')
    ax2.fill_between(x_pos, lci_vals, uci_vals,
                     alpha=0.12, color='#757575',
                     label=f'{int(WH_CONF*100)}% CI')
    ax2.set_ylabel('ln(Energy)', fontsize=14, color='#1565C0')
    ax2.tick_params(axis='y', labelcolor='#1565C0', labelsize=12)

    ax.text(-0.08, 0.98, plabel, transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='bottom', ha='left')
    ax.set_title(feat, fontsize=14, fontweight='bold')
    ax.spines['top'].set_visible(False)

    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=12, loc='upper center',
              ncol=2, framealpha=0.8)

plt.tight_layout()
plt.savefig(f'{OUTPUT_PREFIX}_IMF_validation.png',
            dpi=180, facecolor='white', transparent=False)
plt.close()
print(f"\n✅ 验证图已保存: {OUTPUT_PREFIX}_IMF_validation.png")

# ══════════════════════════════════════════════════════════
# 6. 可视化
# ══════════════════════════════════════════════════════════
t = np.arange(len(train_df))
for feat in FEATURES:
    imfs   = results[feat]['imfs']
    signal = results[feat]['signal']
    val_df = val_results[feat]
    n_comp = len(imfs)

    fig, axes = plt.subplots(n_comp + 1, 1,
                              figsize=(12, 2.0 * (n_comp + 1)),
                              facecolor='white')

    axes[0].plot(t, signal, color='#1565C0', linewidth=2)
    axes[0].set_ylabel('Original', fontsize=26, labelpad=14)
    axes[0].set_xlim(t[0], t[-1])
    for sp in axes[0].spines.values():
        sp.set_visible(True); sp.set_linewidth(2)
    axes[0].tick_params(axis='x', which='both', bottom=False, labelbottom=False)
    axes[0].tick_params(axis='y', labelsize=22)
    axes[0].yaxis.set_major_locator(plt.MaxNLocator(3))
    axes[0].grid(False)

    colors = plt.cm.tab10(np.linspace(0, 0.9, n_comp))
    for i, (imf, c) in enumerate(zip(imfs, colors)):
        ax    = axes[i + 1]
        label = f'IMF{i+1}' if i < n_comp - 1 else 'Residual'
        keep_arr = val_df[val_df['IMF'] == label]['Keep'].values
        kept     = bool(keep_arr[0]) if len(keep_arr) > 0 else True

        plot_color = c       if kept else '#BDBDBD'
        plot_ls    = '-'     if kept else '--'
        plot_lw    = 1.5     if kept else 1.2
        ylabel_txt = label   if kept else f'{label} [Noise]'

        ax.plot(t, imf, color=plot_color, linewidth=plot_lw, linestyle=plot_ls)
        ax.set_ylabel(ylabel_txt, fontsize=20, labelpad=10)
        ax.set_xlim(t[0], t[-1])
        for sp in ax.spines.values():
            sp.set_visible(True); sp.set_linewidth(2)
        ax.grid(False)
        ax.yaxis.set_major_locator(plt.MaxNLocator(3))
        ax.tick_params(axis='y', labelsize=22)
        if i < n_comp - 1:
            ax.tick_params(axis='x', which='both',
                           bottom=False, labelbottom=False)
        else:
            ax.tick_params(axis='x', labelsize=22)
        axes[-1].set_xlabel('Cycle Number', fontsize=26)

    fig.align_ylabels(axes)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_PREFIX}_ICEEMDAN_{feat}.png',
                dpi=180, facecolor='white', transparent=False)
    plt.close()
    print(f"  ✅ 分解图已保存: {OUTPUT_PREFIX}_ICEEMDAN_{feat}.png")

# ══════════════════════════════════════════════════════════
# 7. 
# ══════════════════════════════════════════════════════════
images    = [Image.open(f'{OUTPUT_PREFIX}_ICEEMDAN_{feat}.png')
             for feat in FEATURES]
w, h      = images[0].width, images[0].height
combined  = Image.new('RGB', (w*2, h*2), (255, 255, 255))
positions = [(0, 0), (w, 0), (0, h), (w, h)]
for img, pos in zip(images, positions):
    combined.paste(img.resize((w, h)), pos)

fig2, ax2 = plt.subplots(figsize=(w*2/100, h*2/100), facecolor='white')
ax2.imshow(np.array(combined))
ax2.axis('off')
panel_lbls = ['(a)', '(b)', '(c)', '(d)']
lbl_pos    = [(0, 0), (w, 0), (0, h), (w, h)]
for lbl, (xoff, yoff) in zip(panel_lbls, lbl_pos):
    ax2.text(xoff + w*0.02, yoff - h*0.00, lbl,
             fontsize=48, fontweight='bold', color='black',
             transform=ax2.transData, va='top', ha='left')
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.savefig(f'{OUTPUT_PREFIX}_ICEEMDAN_combined.png',
            dpi=180, bbox_inches='tight',
            facecolor='white', transparent=False)
plt.close()
print(f"✅ 拼合图已保存: {OUTPUT_PREFIX}_ICEEMDAN_combined.png")

# ══════════════════════════════════════════════════════════
# 8. 导出到 Excel
# ══════════════════════════════════════════════════════════
with pd.ExcelWriter(f'{OUTPUT_PREFIX}_ICEEMDAN结果.xlsx',
                    engine='openpyxl') as writer:

    for feat in FEATURES:
        imfs   = results[feat]['imfs']
        n_comp = len(imfs)
        cols   = {cycle_col: train_df[cycle_col].values}
        for i, imf in enumerate(imfs):
            lbl = f'IMF{i+1}' if i < n_comp-1 else 'Residual'
            cols[lbl] = imf
        pd.DataFrame(cols).to_excel(writer, sheet_name=f'{feat}_IMFs', index=False)

    all_val = []
    for feat in FEATURES:
        tmp = val_results[feat].copy()
        tmp.insert(0, 'Feature', feat)
        all_val.append(tmp)
    pd.concat(all_val, ignore_index=True)\
      .to_excel(writer, sheet_name='统计验证结果', index=False)

    energy_rows = []
    for feat in FEATURES:
        imfs   = results[feat]['imfs']
        n_comp = len(imfs)
        energies = [np.sum(imf**2) for imf in imfs]
        total_e  = sum(energies)
        for i, (imf, e) in enumerate(zip(imfs, energies)):
            lbl = f'IMF{i+1}' if i < n_comp-1 else 'Residual'
            energy_rows.append({
                'Feature': feat, 'Component': lbl,
                'Energy': round(e, 6),
                'Energy_Ratio%': round(e/total_e*100, 4),
            })
    pd.DataFrame(energy_rows).to_excel(writer, sheet_name='能量汇总', index=False)

    pd.DataFrame({
        '参数': ['总循环数', '训练集', '测试集', '集成次数I',
                  '加噪系数ε', '最大IMF数', '筛分阈值SD',
                  'LB滞后阶数', '显著性水平α', 'WH置信水平'],
        '值':   [len(df), len(train_df), len(test_df),
                  ENSEMBLE_SIZE, NOISE_STD, MAX_IMF, SD_THRESHOLD,
                  LB_LAGS, LB_ALPHA, WH_CONF]
    }).to_excel(writer, sheet_name='实验设置', index=False)

print(f"✅ Excel已保存: {OUTPUT_PREFIX}_ICEEMDAN结果.xlsx")
print("\n分解与验证汇总:")
for feat in FEATURES:
    vdf     = val_results[feat]
    kept    = vdf[vdf['Keep']]['IMF'].tolist()
    dropped = vdf[~vdf['Keep']]['IMF'].tolist()
    print(f"  {feat}: 保留 {kept}  |  剔除 {dropped if dropped else '无'}")